# Whose Evidence Does the Model Believe? — FULL run

All 480 stimuli (20 facts x 3 source pairs x who-affirms x order x 2 languages).
~35-45 min on a free T4. Run this only after the pilot confirms source
sensitivity (see run_source_deference.ipynb). Produces the CSV; statistics
are computed locally with analysis/... on your machine.

**Before running:** Runtime -> Change runtime type -> **GPU (T4)**.

In [ ]:
!git clone --depth 1 https://github.com/anthropics/jacobian-lens.git
!git clone --depth 1 https://github.com/mleyvaz/jspace-epistemic-lens.git
%cd jacobian-lens
!pip install -q -e .
!nvidia-smi | head -12

In [ ]:
import torch, transformers, jlens
jlens.configure_logging()
MODEL_NAME = "Qwen/Qwen3.5-4B"
hf_model = transformers.AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16).cuda()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)
lens = jlens.JacobianLens.from_pretrained(
    "neuronpedia/jacobian-lens",
    filename="qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
    revision="qwen-n1000",
)
print("layers:", model.n_layers)

In [ ]:
import json, sys, pandas as pd
sys.path.append("/content/jspace-epistemic-lens/src")
import jspace
lexids = jspace.build_lexicons(tokenizer)
with open("/content/jspace-epistemic-lens/experiments/source_deference/stimuli_source_deference.json") as f:
    STIM = json.load(f)
print("total stimuli:", len(STIM))

LAYERS = list(range(18, 31))
rows = []
for k, r in enumerate(STIM):
    jl, ml, _ = lens.apply(model, r["prompt"], layers=LAYERS, positions=[-2])
    for L in LAYERS:
        m, ent = jspace.log_masses(jl[L][0], lexids)
        rows.append(dict(id=r["id"], lang=r["lang"], fact_id=r["fact_id"], pair=r["pair"],
                         aff_type=r["aff_type"], den_type=r["den_type"], aff_first=r["aff_first"],
                         layer=L, ent=ent, log_T=m["T"], log_I=m["I"], log_F=m["F"]))
    if (k + 1) % 40 == 0:
        print(f"{k+1}/{len(STIM)}")
df = pd.DataFrame(rows)
df["B"] = df.log_F - df.log_T
df.to_csv("source_deference_full.csv", index=False)
print("saved:", df.shape)
from google.colab import files
files.download("source_deference_full.csv")

Then, on your machine, from the repo root:
`python experiments/source_deference/analyze.py`  (put the CSV in that folder).